# Model-Based RL: Learn One Model, Solve Many Tasks (CEM + MPC)

Model-free RL learns a value function or policy directly from experience. **Model-based
RL** instead learns a *simulator* of the environment — a **dynamics model**
$f_\phi(s,a)\approx s'$ — and then **plans** with it: imagine action sequences, roll
them through the model, and pick the one with the most predicted reward.

The dynamics model doesn't care about *what task* you're solving — it just predicts
physics. That is the point of this lab: we learn **one** dynamics model from
task-agnostic data (no reward at all), then solve **several different tasks** with it —
swing-up *and* spin-at-a-target-velocity — just by swapping the reward and re-planning,
**with no new environment interaction**. A model-free agent would have to retrain from
scratch for each task.

We build the whole thing on **Pendulum-v1**:

1. **Learn the dynamics** $f_\phi(s,a)\to\Delta s$ by supervised regression (MSE) from
   random, reward-free data.
2. **Plan** with it — **random shooting** refined into the **Cross-Entropy Method
   (CEM)** — run as **Model Predictive Control (MPC)**.
3. **Reuse** the same model for new goals; see where it **fails** (the "cliff"); and
   check **honestly** how it compares to a model-free method (SAC) on sample efficiency.

## Setup

In [ ]:
# Fast dependency install with uv (https://docs.astral.sh/uv).
import sys, os
%pip install -q uv
_target = "" if (sys.prefix != sys.base_prefix or os.environ.get("VIRTUAL_ENV")) else "--system"
!uv pip install -q {_target} --python "{sys.executable}" "gymnasium[classic-control]" imageio matplotlib torch stable-baselines3

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

import gymnasium as gym
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

device = torch.device("cpu")   # tiny model + vectorized CEM -> CPU is fine
print("device:", device)

## The environment: Pendulum-v1

A pendulum that starts at a random angle. The engine is too weak to lift it directly,
so any task that involves getting energy into the system requires *pumping* by swinging.
Details [here](https://gymnasium.farama.org/environments/classic_control/pendulum/).

- **Observation** (3 numbers): $[\cos\theta,\ \sin\theta,\ \dot\theta]$, with $\theta=0$ upright.
- **Action** (1 number): a torque in $[-2, 2]$.

![Pendulum](https://gymnasium.farama.org/_images/pendulum.gif)

## Tasks = reward functions (the dynamics don't change)

A **task** is just a reward. We only ever learn the *dynamics*; the reward for any task
is a known function of the state, so the planner can score plans for **any** task with
the **same** model. We define two very different tasks:

- **Swing-up** — get the pole upright and hold it: reward
  $-(\theta^2 + 0.1\,\dot\theta^2 + 0.001\,a^2)$ (Pendulum's usual reward).
- **Spin** — keep the pole rotating at a target angular velocity $v$: reward
  $-((\dot\theta - v)^2 + 0.001\,a^2)$.

In [ ]:
def angle(obs):
    return torch.atan2(obs[..., 1], obs[..., 0])          # theta in [-pi, pi], 0 = upright

def reward_upright(obs, action):
    return -(angle(obs) ** 2 + 0.1 * obs[..., 2] ** 2 + 0.001 * action ** 2)

def reward_spin(obs, action, target_velocity):
    return -((obs[..., 2] - target_velocity) ** 2 + 0.001 * action ** 2)

## Learning a dynamics model

A small MLP predicts the **change** in the observation, $f_\phi(s,a)\approx \Delta s$
(predicting the delta is easier than the raw next state). We standardize its I/O by the
dataset statistics and fit it by **mean-squared error** — plain supervised regression on
logged transitions $(s,a,s')$, using **no rewards** at all.

In [ ]:
class DynamicsModel(nn.Module):
    def __init__(self, obs_dim=3, act_dim=1, h=200):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(obs_dim + act_dim, h), nn.ReLU(),
                                 nn.Linear(h, h), nn.ReLU(), nn.Linear(h, obs_dim))
        self.register_buffer("x_mean", torch.zeros(obs_dim + act_dim))
        self.register_buffer("x_std",  torch.ones(obs_dim + act_dim))
        self.register_buffer("y_mean", torch.zeros(obs_dim))
        self.register_buffer("y_std",  torch.ones(obs_dim))

    def forward(self, obs, a):
        x = torch.cat([obs, a], dim=-1)
        delta = self.net((x - self.x_mean) / self.x_std) * self.y_std + self.y_mean
        return obs + delta          # predicted next observation


def fit_dynamics(model, data, epochs, batch_size=256, lr=1e-3):
    observations = torch.tensor(np.array([transition[0] for transition in data]), dtype=torch.float32)
    actions      = torch.tensor(np.array([transition[1] for transition in data]), dtype=torch.float32)
    next_obs     = torch.tensor(np.array([transition[2] for transition in data]), dtype=torch.float32)

    model_inputs  = torch.cat([observations, actions], dim=1)
    target_deltas = next_obs - observations                    # the model predicts the CHANGE in obs
    model.x_mean, model.x_std = model_inputs.mean(0),  model_inputs.std(0)  + 1e-6
    model.y_mean, model.y_std = target_deltas.mean(0), target_deltas.std(0) + 1e-6

    optimizer = optim.Adam(model.parameters(), lr=lr)
    for _ in range(epochs):
        shuffled = torch.randperm(len(data))
        for start in range(0, len(data), batch_size):
            batch = shuffled[start:start + batch_size]
            predicted_next_obs = model(observations[batch], actions[batch])
            # regression loss: MSE between the predicted next observation and the true one
            loss = ((predicted_next_obs - next_obs[batch]) ** 2).mean()
            optimizer.zero_grad(); loss.backward(); optimizer.step()
    return loss.detach().item()

## Planning: roll the model forward, keep the best sequence

**Planning** = pick the action sequence whose model roll-out earns the most reward
*for the current task's reward function*:
$$a_{t:t+H}^\star = \arg\max_{a_{t:t+H}} \sum_{k=0}^{H} r(s_{t+k}, a_{t+k}),
  \qquad s_{t+k+1} = f_\phi(s_{t+k}, a_{t+k}).$$

**Random shooting** samples many sequences and keeps the best. **CEM** improves on it:
sample from a Gaussian over action sequences, keep the top-$k$ **elites**, refit the
Gaussian to them, repeat. Both take the reward function as an argument — that is what
lets one model serve many tasks.

In [ ]:
@torch.no_grad()                                 # planning never needs gradients
def rollout_return(model, start_obs, action_sequences, reward_fn):
    """Predicted return of each candidate action sequence under `reward_fn`.

    action_sequences: (num_candidates, horizon) tensor. Returns (num_candidates,) returns.
    """
    num_candidates, horizon = action_sequences.shape
    obs = start_obs.unsqueeze(0).repeat(num_candidates, 1)     # every candidate starts at start_obs
    returns = torch.zeros(num_candidates)
    for step in range(horizon):
        action = action_sequences[:, step:step + 1]           # (num_candidates, 1)
        # add this step's reward for the CURRENT task, then advance the state through the model
        returns = returns + reward_fn(obs, action[:, 0])
        obs = model(obs, action)
    return returns

In [ ]:
def cem_plan(model, state, warm_start, cfg, reward_fn):
    """One CEM optimization of a horizon-length action plan starting from `state`, for
    the task defined by `reward_fn`. `warm_start` seeds the search. Returns the plan."""
    start_obs = torch.tensor(state, dtype=torch.float32)
    mean = torch.tensor(warm_start, dtype=torch.float32).clone()   # mean of the sampling Gaussian
    std = torch.ones(cfg["H"]) * cfg["init_std"]                   # its per-step std
    num_elite = max(2, int(cfg["N"] * cfg["elite_frac"]))
    for _ in range(cfg["cem_iters"]):
        action_sequences = (mean + std * torch.randn(cfg["N"], cfg["H"])).clamp(-2.0, 2.0)
        returns = rollout_return(model, start_obs, action_sequences, reward_fn)
        elite_indices = torch.topk(returns, num_elite).indices
        elites = action_sequences[elite_indices]
        # CEM step: refit the sampling distribution (mean & std) to the elite sequences
        mean = elites.mean(0)
        std = elites.std(0) + 1e-4
    return mean.numpy()          # the planned action sequence (length H)

## Model Predictive Control (MPC)

A full open-loop plan is fragile — model errors compound over the horizon. **MPC** plans
a horizon, executes **only the first action**, then re-plans from the state it actually
lands in (warm-starting from the previous plan).

In [ ]:
def run_mpc_episode(env, model, cfg, reward_fn, seed=None, render=False):
    obs, _ = env.reset(seed=seed)
    plan = np.zeros(cfg["H"])          # warm-start plan
    total_reward, frames, velocities = 0.0, [], []
    for _ in range(200):
        if render:
            frames.append(env.render())
        plan = cem_plan(model, obs, plan, cfg, reward_fn)
        action = float(plan[0])
        obs, reward, terminated, truncated, _ = env.step([action])
        total_reward += reward
        velocities.append(obs[2])
        plan = np.append(plan[1:], 0.0)                    # shift the plan forward (warm start)
        if terminated or truncated:
            break
    held_velocity = float(np.mean(velocities[-80:]))       # angular velocity held at the end
    return total_reward, frames, held_velocity

## Configuration — experiment here

In [ ]:
config = {
    # --- CEM planner ---
    "H":          20,     # planning horizon (steps looked ahead)
    "N":          300,    # candidate action sequences sampled per CEM iteration
    "cem_iters":  5,      # CEM refinement iterations per decision
    "elite_frac": 0.1,    # fraction of candidates kept as elites
    "init_std":   1.0,    # initial per-step action std (action range is [-2, 2])
    # --- data / model ---
    "data_episodes": 40,  # episodes of task-agnostic random data
    "model_epochs":  100, # dynamics-model training epochs
}

## Learn ONE dynamics model from task-agnostic data

We collect random (full-torque) transitions — no policy, no reward, just diverse physics
— and fit the dynamics model once. This is the only real environment interaction the
model-based agent uses; everything below reuses this single model.

In [ ]:
def collect_random(env, n_episodes, seed=0):
    transitions = []
    for e in range(n_episodes):
        obs, _ = env.reset(seed=seed + e)
        for _ in range(200):
            action = np.array([np.random.uniform(-2, 2)], dtype=np.float32)   # random torque
            next_obs, _, terminated, truncated, _ = env.step(action)
            transitions.append((obs.copy(), action.copy(), next_obs.copy()))
            obs = next_obs
            if terminated or truncated:
                break
    return transitions

np.random.seed(0)
torch.manual_seed(0)
env = gym.make("Pendulum-v1")

data = collect_random(env, config["data_episodes"])
model_steps = len(data)                         # real environment steps spent on the model
model = DynamicsModel().to(device)
mse = fit_dynamics(model, data, config["model_epochs"])
print(f"learned dynamics from {model_steps} random transitions | final MSE = {mse:.5f}")

## Task 1 — swing up

Plan with `reward_upright`. Return near $-1200$ is a failure (random), $\gtrsim -200$ is
a solved swing-up.

In [ ]:
swingup_return, _, swingup_velocity = run_mpc_episode(env, model, config, reward_upright, seed=0)
print(f"swing-up return: {swingup_return:.1f}  (held velocity {swingup_velocity:+.1f}, ~0 = balanced upright)")

## The payoff — one model, many tasks

Now the point of the whole lab. **Without touching the environment again**, we swap the
reward to `reward_spin` and re-plan. The *same* model now solves a completely different
task: keep the pole spinning at a target speed — slow or fast. A model-free agent
trained to swing up could not do this — it would have to retrain from scratch.

In [ ]:
_, _, fast_velocity = run_mpc_episode(env, model, config, lambda obs, action: reward_spin(obs, action, 6.0), seed=0)
_, _, slow_velocity = run_mpc_episode(env, model, config, lambda obs, action: reward_spin(obs, action, 3.0), seed=0)

print("the SAME model, three different tasks -- all with zero new environment steps:")
print(f"  swing-up    : held velocity {swingup_velocity:+.1f}   (target 0, balanced upright)")
print(f"  spin (fast) : held velocity {fast_velocity:+.1f}   (target +6)")
print(f"  spin (slow) : held velocity {slow_velocity:+.1f}   (target +3)")

## Watch one model do three different things

In [ ]:
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")   # headless pygame rendering
import imageio.v2 as imageio
from IPython.display import Image, display

os.makedirs("video", exist_ok=True)
render_env = gym.make("Pendulum-v1", render_mode="rgb_array")
tasks = [("swing-up",  reward_upright),
         ("spin fast", lambda obs, action: reward_spin(obs, action, 6.0)),
         ("spin slow", lambda obs, action: reward_spin(obs, action, 3.0))]
for name, reward_fn in tasks:
    _, frames, _ = run_mpc_episode(render_env, model, config, reward_fn, seed=0, render=True)
    filename = "video/mbrl_" + name.replace(" ", "_").replace("-", "_") + ".gif"
    imageio.mimsave(filename, frames, fps=30, loop=0)
    print(name + ":")
    display(Image(filename=filename))
render_env.close()

## The catch: a model is only right where it has data (the "cliff")

Why collect *diverse* data? Because a model is only trustworthy **on its data
distribution**. To see it, we train a second model on **weak-torque** data — the pole
only ever flails near the *bottom* and never gets near the top. Ask *that* model to plan
the **swing-up** (which lives up near upright, exactly where it has no data) and it is
confidently wrong: the planner chases a fantasy and the pole never balances.

This is the **distribution-shift "cliff"** from the lecture. In the model-based setting
we could fix it by going out to collect the missing data. In the next lab (**offline
RL**) that fix is *forbidden* — the same problem, with your hands tied.

In [ ]:
def collect_weak(env, n_episodes, seed=0, torque_scale=0.3):
    transitions = []
    for e in range(n_episodes):
        obs, _ = env.reset(seed=seed + e)
        for _ in range(200):
            action = np.array([np.random.uniform(-2, 2) * torque_scale], dtype=np.float32)  # weak torque
            next_obs, _, terminated, truncated, _ = env.step(action)
            transitions.append((obs.copy(), action.copy(), next_obs.copy()))
            obs = next_obs
            if terminated or truncated:
                break
    return transitions

weak_data = collect_weak(env, config["data_episodes"])
narrow_model = DynamicsModel().to(device)
fit_dynamics(narrow_model, weak_data, config["model_epochs"])

narrow_return, _, _ = run_mpc_episode(env, narrow_model, config, reward_upright, seed=0)
print(f"swing-up return, DIVERSE-data model : {swingup_return:.0f}   (solved)")
print(f"swing-up return, WEAK-data model    : {narrow_return:.0f}   <- cliff: never saw the top, can't balance there")

## Is it more sample-efficient? An honest look vs SAC

On a *simple* task like swing-up, a strong model-free method is already very
sample-efficient — so model-based does **not** win by a wide margin *per task* here (the
lecture's dramatic model-based sample-efficiency wins come on **hard, high-dimensional**
tasks). We check with **SAC** (Stable-Baselines3), evaluating its greedy policy as it
trains.

The honest picture is about **amortization**: our one model cost ~$8{,}000$ real steps
and then solved *three* tasks for free; SAC must spend its budget **again for every new
task**. So the model-based cost is flat as tasks pile up, while the model-free cost grows.

In [ ]:
from stable_baselines3 import SAC
from stable_baselines3.common.evaluation import evaluate_policy

sac = SAC("MlpPolicy", gym.make("Pendulum-v1"), verbose=0, seed=0)

sac_steps, sac_returns = [], []
for chunk in range(1, 5):                        # train in 2000-step chunks, up to 8000
    sac.learn(2000, reset_num_timesteps=False)
    mean_return, _ = evaluate_policy(sac, gym.make("Pendulum-v1"), n_eval_episodes=5)
    sac_steps.append(chunk * 2000)
    sac_returns.append(float(mean_return))

# how many real steps did SAC need to first reach a solved swing-up (return above -200)?
sac_solve = sac_steps[-1]
for steps, ret in zip(sac_steps, sac_returns):
    if ret > -200:
        sac_solve = steps
        break
print(f"SAC solves swing-up in ~{sac_solve} real steps (one task)")

In [ ]:
num_tasks = np.arange(1, 6)
model_based_cost = np.full(len(num_tasks), model_steps)   # one model, reused for every task
model_free_cost  = sac_solve * num_tasks                  # SAC retrains from scratch per task

fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4))

left.plot(sac_steps, sac_returns, "s-", color="tab:red", label="SAC (per task)")
left.axhline(-200, color="green", ls="--", alpha=0.6, label="~solved")
left.axvline(model_steps, color="tab:blue", ls=":", label=f"model's data budget ({model_steps})")
left.set_xlabel("real env steps")
left.set_ylabel("greedy return")
left.set_title("Single task (swing-up)")
left.legend()
left.grid(alpha=0.3)

right.plot(num_tasks, model_based_cost, "o-", color="tab:blue", label="model-based (one model, reused)")
right.plot(num_tasks, model_free_cost,  "s-", color="tab:red",  label="model-free SAC (retrain each)")
right.set_xlabel("number of tasks solved")
right.set_ylabel("total real env steps")
right.set_title("Model-free cost grows per task; model-based is flat")
right.legend()
right.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Takeaways

- **Why build a model? It is task-agnostic.** The dynamics don't depend on the reward, so
  **one** model — learned once, from reward-free data — solves *many* tasks (swing-up,
  spin-up, spin-down) just by re-planning. A model-free policy is tied to the single
  reward it trained on.
- **How it works:** fit a dynamics model by supervised regression, then plan through it
  (random shooting → CEM) under **MPC**.
- **Sample efficiency, honestly:** on a simple task the per-task win over SAC is small;
  the real saving is **amortization** across tasks (and it becomes dramatic on hard,
  high-dimensional tasks — the lecture's dexterous-manipulation case study).
- **The cliff:** a model is only trustworthy on its data distribution. Off it, the
  planner chases a fantasy. That same off-distribution problem returns in **offline RL**
  (next lab), where you are *forbidden* from collecting the missing data.
- Modern descendants plan with a learned model (**MuZero**) or learn a policy inside it
  (**Dreamer**); this CEM-MPC loop is the smallest version that works.